# 6-1. OpenFermion の基礎

参考: Quantum Native Dojo の「6-1. OpenFermionの使い方」を、現在の `openfermion` 系 API で動かしやすいように整理したノートブックです。

このノートブックでは、水素分子 $\mathrm{H}_2$ を例にして、

1. 分子情報の定義
2. PySCF による量子化学計算
3. 1電子積分・2電子積分の確認
4. 第二量子化ハミルトニアンの生成
5. Jordan-Wigner / Bravyi-Kitaev 変換

までを、セルを分けて順に確認します。

## 0. 依存関係の確認

このノートブックの実行には少なくとも以下が必要です。

- `openfermion`
- `openfermionpyscf`
- `pyscf`
- `matplotlib`

不足している場合は、下のセルの案内に従ってインストールしてください。

In [ ]:
import importlib.util

required_modules = ["openfermion", "openfermionpyscf", "pyscf", "matplotlib", "numpy"]
missing_modules = [name for name in required_modules if importlib.util.find_spec(name) is None]

if missing_modules:
    print("Missing modules:", ", ".join(missing_modules))
    print("Install example:")
    print("pip install openfermion openfermionpyscf pyscf matplotlib")
else:
    print("All required modules were found.")

## 1. ライブラリの import

OpenFermion は現行版ではトップレベルから主要 API を import する書き方が扱いやすいので、その形にそろえます。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from openfermion import (
    MolecularData,
    get_fermion_operator,
    jordan_wigner,
    bravyi_kitaev,
    get_sparse_operator,
)
from openfermion.linalg import get_ground_state
from openfermionpyscf import run_pyscf

## 2. 水素分子 $\mathrm{H}_2$ の設定

以下の変数で分子を定義します。

- `basis`: 基底関数系
- `multiplicity`: スピン多重度
- `charge`: 全電荷
- `distance`: 原子間距離
- `geometry`: 原子核配置
- `description`: 計算結果識別用の文字列

In [ ]:
basis = "sto-3g"
multiplicity = 1
charge = 0
distance = 0.65
geometry = [("H", (0.0, 0.0, 0.0)), ("H", (0.0, 0.0, distance))]
description = f"bond_length_{distance:.2f}"

molecule = MolecularData(geometry, basis, multiplicity, charge, description)

print("geometry =", geometry)
print("basis =", basis)
print("multiplicity =", multiplicity)
print("charge =", charge)

## 3. PySCF による量子化学計算

ここでは SCF と FCI を実行します。

- SCF: Hartree-Fock 計算
- FCI: Full-CI 計算

FCI はこの小さな系では厳密な基底エネルギーに対応します。

In [ ]:
molecule = run_pyscf(molecule, run_scf=True, run_fci=True)

print("PySCF calculation finished.")

## 4. Hartree-Fock エネルギーと Full-CI エネルギー

In [ ]:
print(f"HF energy  : {molecule.hf_energy:.12f} Hartree")
print(f"FCI energy : {molecule.fci_energy:.12f} Hartree")

## 5. 1電子積分 $h_{ij}$ と 2電子積分 $h_{ijkl}$

OpenFermion の `MolecularData` には、分子軌道基底での積分値が保存されています。

In [ ]:
print("one-body integrals shape:", molecule.one_body_integrals.shape)
print(molecule.one_body_integrals)

In [ ]:
print("two-body integrals shape:", molecule.two_body_integrals.shape)
print(molecule.two_body_integrals)

## 6. 第二量子化形式のハミルトニアン

まず `get_molecular_hamiltonian()` で分子ハミルトニアンを得て、そのあと `get_fermion_operator()` で FermionOperator に変換します。

In [ ]:
molecular_hamiltonian = molecule.get_molecular_hamiltonian()
fermion_hamiltonian = get_fermion_operator(molecular_hamiltonian)

print("Molecular Hamiltonian (InteractionOperator):")
print(molecular_hamiltonian)

print("\nFermionOperator form:")
print(fermion_hamiltonian)

## 7. 量子コンピュータで扱いやすい演算子への変換

FermionOperator を、量子ビット演算子へ写像します。ここでは 2 種類を確認します。

- Jordan-Wigner 変換
- Bravyi-Kitaev 変換

In [ ]:
jw_hamiltonian = jordan_wigner(fermion_hamiltonian)
bk_hamiltonian = bravyi_kitaev(fermion_hamiltonian)

print("Jordan-Wigner Hamiltonian:")
print(jw_hamiltonian)

print("\nBravyi-Kitaev Hamiltonian:")
print(bk_hamiltonian)

## 8. 疎行列化して基底エネルギーを確認する

量子ビットハミルトニアンを疎行列に変換し、数値的に基底状態エネルギーを確認します。FCI エネルギーと一致するはずです。

In [ ]:
jw_sparse = get_sparse_operator(jw_hamiltonian)
ground_energy, ground_state = get_ground_state(jw_sparse)

print(f"Ground-state energy from JW sparse operator: {ground_energy:.12f} Hartree")
print(f"Reference FCI energy                     : {molecule.fci_energy:.12f} Hartree")
print(f"Absolute difference                      : {abs(ground_energy - molecule.fci_energy):.3e} Hartree")

## 9. 原子間距離を変えたときのエネルギー曲線

モジュールを分けた構成にしたので、距離依存を確認する小さな関数も独立して定義できます。

In [ ]:
def compute_h2_energy(distance, basis="sto-3g", multiplicity=1, charge=0):
    geometry = [("H", (0.0, 0.0, 0.0)), ("H", (0.0, 0.0, float(distance)))]
    description = f"bond_length_{distance:.2f}"
    mol = MolecularData(geometry, basis, multiplicity, charge, description)
    mol = run_pyscf(mol, run_scf=True, run_fci=True)
    return {
        "distance": distance,
        "hf_energy": mol.hf_energy,
        "fci_energy": mol.fci_energy,
    }

In [ ]:
distances = np.linspace(0.4, 2.0, 9)
results = [compute_h2_energy(d) for d in distances]

hf_energies = [item["hf_energy"] for item in results]
fci_energies = [item["fci_energy"] for item in results]

plt.figure(figsize=(7, 4))
plt.plot(distances, hf_energies, "o-", label="Hartree-Fock")
plt.plot(distances, fci_energies, "s-", label="Full-CI")
plt.xlabel("H-H distance (Angstrom)")
plt.ylabel("Energy (Hartree)")
plt.title("H2 energy curve in STO-3G")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## 10. まとめ

このノートブックでは、水素分子の量子化学ハミルトニアンを OpenFermion と PySCF で生成し、第二量子化形式から量子ビット演算子へ写像する流れを確認しました。

次の段階では、この量子ビットハミルトニアンを使って VQE などの変分アルゴリズムを実装できます。